# RAM-optimized baseline

Основная идея: не загружать 30.6 млн строк × 18 колонок в память, если для текущей модели реально нужны только `event_date`, `user_id` и `gmv`.

Сохраняется логика исходного ноутбука:
- сортировка по `user_id`, `event_date`;
- признаки `day`, `month`, `weekday`;
- `target` = rolling sum `gmv` за 30 дней с `closed="both"`;
- chronological split по строкам 75/25;
- `log1p(target)`;
- CatBoost на GPU.

Главное изменение — column pruning, downcast типов и отсутствие лишних pandas/sklearn-копий.


In [1]:
import gc
import random

import numpy as np
import polars as pl
from catboost import CatBoostRegressor, Pool

pl.Config.set_tbl_rows(6)

PARQUET_PATH = "data/train.parquet"
TEST_SIZE = 0.25
SEED = 90



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Projects\e-cup-26-search\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Projects\e-cup-26-search\.venv\Lib\site-packages\traitlets\config\application.py", line 1080, in launch_instance
    app.start()
  File "c:\Projects\e-cup-26-search\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 807, in start
    self.io_loop.start()
  File "c:\Projects

AttributeError: _ARRAY_API not found

## 1. Читаем только необходимые колонки

В исходнике читались все 18 колонок. Для фактически используемых фич нужны только 3.

`scan_parquet` откладывает чтение до `collect()`, поэтому Polars может протолкнуть `select` в Parquet и не материализовать ненужные колонки.


In [2]:
needed_cols = ["event_date", "user_id", "gmv"]

lf = (
    pl.scan_parquet(PARQUET_PATH)
    .select(needed_cols)
    .with_columns(
        pl.col("user_id").cast(pl.UInt32),
        pl.col("gmv").cast(pl.Float32),
    )
)

print("Input schema:", lf.collect_schema())


Input schema: Schema({'event_date': Date, 'user_id': UInt32, 'gmv': Float32})


## 2. Feature engineering одним lazy-пайплайном

Не создаём промежуточные DataFrame. `month` и `weekday` оставляем числовыми: для CatBoost их можно передать как categorical features, а `UInt8` занимает намного меньше памяти, чем строки вроде `"Aug"` / `"Fri"`.

`target` считается с той же семантикой, что и в исходном ноутбуке.


In [3]:
features = (
    lf
    .sort(["user_id", "event_date"])
    .with_columns(
        pl.col("event_date").dt.day().cast(pl.UInt8).alias("day"),
        pl.col("event_date").dt.month().cast(pl.UInt8).alias("month"),
        # Monday=1 ... Sunday=7
        pl.col("event_date").dt.weekday().cast(pl.UInt8).alias("weekday"),
    )
    .with_columns(
        pl.col("gmv")
        .sum()
        .rolling(
            "event_date",
            period="30d",
            closed="both",
        )
        .over("user_id")
        .alias("target")
    )
    .select(["user_id", "day", "month", "weekday", "target"])
    .collect()
)

# После collect исходный lazy-план больше не нужен.
del lf
gc.collect()

print("Features:", features.shape)
print(features.schema)


Features: (30631006, 5)
Schema({'user_id': UInt32, 'day': UInt8, 'month': UInt8, 'weekday': UInt8, 'target': Float32})


## 3. Не держим одновременно исходные данные и фичи

В исходнике `data`, `df_with_target`, `X`, `y`, `X_train`, `X_test`, `y_train`, `y_test` могли одновременно занимать память.

Здесь оставляем один компактный feature frame и создаём только необходимые срезы.


In [4]:
train_len = int(features.height * (1 - TEST_SIZE))

train_df = features.slice(0, train_len)
test_df = features.slice(train_len)

# target нужен как label, но не должен попасть в X.
X_train = train_df.drop("target")
X_test = test_df.drop("target")

y_train = train_df["target"].cast(pl.Float32)
y_test = test_df["target"].cast(pl.Float32)

# log1p в Float32: в 2 раза меньше памяти, чем Float64.
y_train_log = y_train.log1p().cast(pl.Float32)
y_test_log = y_test.log1p().cast(pl.Float32)

del train_df, test_df, y_train, y_test
gc.collect()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


X_train: (22973254, 4)
X_test : (7657752, 4)


## 4. CatBoost без sklearn Pipeline

`StandardScaler` для `day` здесь не нужен: деревья CatBoost не требуют масштабирования.

Также убраны `ColumnTransformer` и промежуточный polars-output sklearn — это лишние преобразования и потенциальные копии на десятках миллионов строк.


In [5]:
CAT_FEATURES = ["user_id", "month", "weekday"]

train_pool = Pool(
    data=X_train,
    label=y_train_log,
    cat_features=CAT_FEATURES,
)

test_pool = Pool(
    data=X_test,
    label=y_test_log,
    cat_features=CAT_FEATURES,
)

del X_train, X_test, y_train_log, y_test_log
gc.collect()


0

## 5. Обучение

Grid search `2 × 3` с `cv=4` из исходника может многократно раздувать RAM/VRAM и время.

Ниже оставлен один запуск CatBoost. Если grid search принципиально нужен, запускайте его отдельно после освобождения всех лишних объектов.


In [6]:
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=SEED,
    task_type="CPU",
    verbose=True,
    # Для CPU можно дополнительно задать used_ram_limit.
    # used_ram_limit="8gb",
)

model.fit(
    train_pool,
    eval_set=test_pool,
    early_stopping_rounds=30,
    verbose=True,
    plot=True
)

del train_pool, test_pool
gc.collect()


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Learning rate set to 0.246381
0:	learn: 2.1021380	test: 2.3349575	best: 2.3349575 (0)	total: 5.3s	remaining: 1h 28m 20s
1:	learn: 1.9577941	test: 2.3558265	best: 2.3349575 (0)	total: 10.5s	remaining: 1h 27m 3s
2:	learn: 1.8671603	test: 2.3729540	best: 2.3349575 (0)	total: 12.2s	remaining: 1h 7m 27s
3:	learn: 1.7215562	test: 2.3891497	best: 2.3349575 (0)	total: 19.1s	remaining: 1h 19m 21s
4:	learn: 1.6300341	test: 2.4516249	best: 2.3349575 (0)	total: 21s	remaining: 1h 9m 45s
5:	learn: 1.5736665	test: 2.5153295	best: 2.3349575 (0)	total: 25.9s	remaining: 1h 11m 26s
6:	learn: 1.5402728	test: 2.5590530	best: 2.3349575 (0)	total: 27.6s	remaining: 1h 5m 21s
7:	learn: 1.5195238	test: 2.6480155	best: 2.3349575 (0)	total: 33.7s	remaining: 1h 9m 41s
8:	learn: 1.5033530	test: 2.7348653	best: 2.3349575 (0)	total: 35.6s	remaining: 1h 5m 23s
9:	learn: 1.4925586	test: 2.8701230	best: 2.3349575 (0)	total: 38.2s	remaining: 1h 2m 58s


KeyboardInterrupt: 

## 6. Если нужен именно grid search

Не держите несколько Pool/DataFrame одновременно. После каждого большого эксперимента удаляйте модель и промежуточные объекты.

Для максимальной экономии RAM лучше подбирать параметры на уменьшенной выборке, а затем один раз обучать финальную модель на всех 30.6 млн строк.


In [7]:
# Пример memory-friendly подбора на небольшой подвыборке:

sample_size = min(2_000_000, features.height)
sample_df = features.sample(n=sample_size, seed=SEED)
sample_X = sample_df.drop("target")
sample_y = sample_df["target"].log1p().cast(pl.Float32)

# sample_pool = Pool(
#     sample_X,
#     label=sample_y,
#     cat_features=CAT_FEATURES,
# )

# search_model = CatBoostRegressor(
#     loss_function="RMSE",
#     eval_metric="RMSE",
#     random_seed=SEED,
#     task_type="CPU",
#     verbose=False,
# )

# search_model.grid_search(
#     {
#         "learning_rate": [0.03, 0.1],
#         "l2_leaf_reg": [1, 3, 5],
#     },
#     X=sample_pool,
#     cv=4,
#     partition_random_seed=SEED,
#     verbose=True,
#     plot=True
# )

# del sample_pool, sample_X, sample_y, sample_df, search_model
# gc.collect()


In [ ]:
SUBMIT_PATH = "sample_submit.csv"
OUT_PATH = "submission.csv"

PRED_DATE = "2026-03-15"

FEATURES = ["user_id", "day", "month", "weekday"]
CAT_FEATURES = ["user_id", "month", "weekday"]


# ---------------------------------------------------------
# 1. sample_submit — именно он определяет порядок user_id
# ---------------------------------------------------------

submit = pl.read_csv(
    SUBMIT_PATH,
    schema_overrides={"user_id": pl.UInt32},
)

# ---------------------------------------------------------
# 2. Признаки для 15 марта 2026
# ---------------------------------------------------------

X_test = (
    submit
    .with_columns(
        pl.lit(PRED_DATE).str.to_date().alias("event_date")
    )
    .with_columns(
        pl.col("event_date").dt.day()
            .cast(pl.UInt8)
            .alias("day"),

        pl.col("event_date").dt.month()
            .cast(pl.UInt8)
            .alias("month"),

        pl.col("event_date").dt.weekday()
            .cast(pl.UInt8)
            .alias("weekday"),
    )
    .select(FEATURES)
)

# ---------------------------------------------------------
# 3. CatBoost Pool
# ---------------------------------------------------------

X_test_pd = X_test.to_pandas()

test_pool = Pool(
    X_test_pd,
    cat_features=CAT_FEATURES,
)

# ---------------------------------------------------------
# 4. Prediction
# ---------------------------------------------------------

pred_log = model.predict(test_pool)
pred = np.expm1(pred_log).astype(np.float64)

# ---------------------------------------------------------
# 5. Submission
# ---------------------------------------------------------

submission = submit.with_columns(
    pl.Series(
        "predict",
        pred,
        dtype=pl.Float64,
    )
)

submission.write_csv(OUT_PATH)

print(submission)
print(f"\nSaved: {OUT_PATH}")